# 01 · Data preparation — HAM10000

**Goal.** Download HAM10000, attach each image to its patient metadata row,
derive the binary (benign/malignant) label, and create **one global
stratified split** that every downstream stage reuses (so the end-to-end test
set in notebook 06 is never seen during any training).

We also **fit the metadata preprocessor on the training split only** and persist
it to `models/preprocessing.json` — the exact same object the Streamlit app
loads, guaranteeing train/serve consistency.

---
*Part of: **A Multimodal, Explainable Hierarchical Framework for Skin-Lesion
Classification with Diffusion-Based Augmentation of Rare Malignancies.***
All training/eval logic lives in the tested modules under `src/`; these
notebooks orchestrate them on a Colab T4.

In [1]:
# --- Install dependencies (Colab already has TF/torch; this pins the extras) ---
# If a package is already present Colab will skip it quickly.
!pip -q install scikit-learn matplotlib kaggle >/dev/null 2>&1

deps ready


In [4]:
import os, sys

try:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_DIR = '/content/drive/MyDrive/Projects/Skin_Cancer_Research_Grant2024'
except Exception:
    PROJECT_DIR = os.path.abspath('..')

SRC       = os.path.join(PROJECT_DIR, 'src')
MODELS_DIR= os.path.join(PROJECT_DIR, 'models')
DATA_DIR  = os.path.join(PROJECT_DIR, 'data')
FIG_DIR   = os.path.join(PROJECT_DIR, 'report', 'figures')
SPLIT_CSV = os.path.join(MODELS_DIR, 'ham_split.csv')
PREP_JSON = os.path.join(MODELS_DIR, 'preprocessing.json')
for d in (MODELS_DIR, DATA_DIR, FIG_DIR):
    os.makedirs(d, exist_ok=True)

assert os.path.isdir(SRC), f"src/ not found at {SRC} — set PROJECT_DIR correctly"
sys.path.insert(0, SRC)
print("PROJECT_DIR =", PROJECT_DIR)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
PROJECT_DIR = /content/drive/MyDrive/Projects/Skin_Cancer_Research_Grant2024


## Download HAM10000 from Kaggle

In [5]:
from google.colab import files

files.upload()
!mkdir -p ~/.kaggle && cp kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json

# ~5k+5k images across two folders + HAM10000_metadata.csv
!kaggle datasets download -d kmader/skin-cancer-mnist-ham10000 -p "$DATA_DIR" --unzip
print("downloaded to:", DATA_DIR)

Saving kaggle.json to kaggle (1).json
Dataset URL: https://www.kaggle.com/datasets/kmader/skin-cancer-mnist-ham10000
License(s): CC-BY-NC-SA-4.0
100% 5.20G/5.20G [00:28<00:00, 194MB/s]

downloaded to /content/drive/MyDrive/Projects/Skin_Cancer_Research_Grant2024/data


In [6]:
import glob, os
meta_csv = glob.glob(os.path.join(DATA_DIR, '**', 'HAM10000_metadata*.csv'), recursive=True)[0]
image_dirs = sorted({os.path.dirname(p) for p in
                     glob.glob(os.path.join(DATA_DIR, '**', '*.jpg'), recursive=True)})
print("metadata:", meta_csv)
print("image dirs:", image_dirs)

metadata: /content/drive/MyDrive/Projects/Skin_Cancer_Research_Grant2024/data/HAM10000_metadata.csv
image dirs: ['/content/drive/MyDrive/Projects/Skin_Cancer_Research_Grant2024/data/HAM10000_images_part_1', '/content/drive/MyDrive/Projects/Skin_Cancer_Research_Grant2024/data/HAM10000_images_part_2', '/content/drive/MyDrive/Projects/Skin_Cancer_Research_Grant2024/data/ham10000_images_part_1', '/content/drive/MyDrive/Projects/Skin_Cancer_Research_Grant2024/data/ham10000_images_part_2']


## Build the dataframe + global split (uses the tested `src/data.py`)

In [7]:
import pandas as pd
import data as D
import metadata as M

df = D.load_ham_dataframe(meta_csv, image_dirs)
print("rows with images:", len(df))
print("\ndx distribution:\n", df['dx'].value_counts())
print("\nbinary distribution:\n", df['binary_label'].value_counts())

df = D.assign_global_split(df, test_size=0.15, val_size=0.15)
print("\nsplit counts:\n", df['split'].value_counts())
df.to_csv(SPLIT_CSV, index=False)
print("\nsaved split ->", SPLIT_CSV)

rows with images: 10015

dx distribution:
 dx
nv       6705
mel      1113
bkl      1099
bcc       514
akiec     327
vasc      142
df        115
Name: count, dtype: int64

binary distribution:
 binary_label
benign       8061
malignant    1954
Name: count, dtype: int64

split counts:
 split
train    7235
test     1503
val      1277
Name: count, dtype: int64

saved split -> /content/drive/MyDrive/Projects/Skin_Cancer_Research_Grant2024/models/ham_split.csv


## Fit + save the metadata preprocessor (**train split only**)
Fitting the age-standardiser and one-hot vocabularies on the training rows only
avoids leaking test statistics into preprocessing.

In [8]:
train_df = df[df['split'] == 'train']
proc = M.MetadataProcessor().fit(train_df)
proc.save(PREP_JSON)
print("metadata feature_dim =", proc.feature_dim)
print("feature names:", proc.feature_names)
print("saved ->", PREP_JSON)

metadata feature_dim = 19
feature names: ['age_z', 'sex=female', 'sex=male', 'sex=unknown', 'loc=abdomen', 'loc=acral', 'loc=back', 'loc=chest', 'loc=ear', 'loc=face', 'loc=foot', 'loc=genital', 'loc=hand', 'loc=lower extremity', 'loc=neck', 'loc=scalp', 'loc=trunk', 'loc=unknown', 'loc=upper extremity']
saved -> /content/drive/MyDrive/Projects/Skin_Cancer_Research_Grant2024/models/preprocessing.json
